# Process IPEDS salary compression data

This notebook reads the IPEDS `Data*.csv` export and creates a tidy salary file for comparing Full Professor and Assistant Professor average salaries.

## Imports and paths

In [1]:
from pathlib import Path

import pandas as pd

process_dir = Path.cwd()
project_dir = process_dir.parent
data_dir = project_dir / "data"
ipeds_dir = process_dir / "6-9-2026---521"

data_files = sorted(ipeds_dir.glob("Data*.csv"))
if len(data_files) != 1:
    raise ValueError(f"Expected exactly one Data*.csv file in {ipeds_dir}, found {len(data_files)}")

label_files = sorted(ipeds_dir.glob("ValueLabels*.csv"))
if len(label_files) != 1:
    raise ValueError(f"Expected exactly one ValueLabels*.csv file in {ipeds_dir}, found {len(label_files)}")

input_path = data_files[0]
labels_path = label_files[0]
output_path = process_dir / "ipeds_salary_compression_tidy.csv"
compression_output_path = process_dir / "compression_ratios.csv"
compression_data_output_path = data_dir / "compression_ratios.csv"

input_path, output_path, compression_output_path, compression_data_output_path

(PosixPath('/Users/mcmcclur/GitHubDocuments/MarksMath/writ/CompensationAtUNCA/_process/6-9-2026---521/Data_6-9-2026---521.csv'),
 PosixPath('/Users/mcmcclur/GitHubDocuments/MarksMath/writ/CompensationAtUNCA/_process/ipeds_salary_compression_tidy.csv'),
 PosixPath('/Users/mcmcclur/GitHubDocuments/MarksMath/writ/CompensationAtUNCA/_process/compression_ratios.csv'),
 PosixPath('/Users/mcmcclur/GitHubDocuments/MarksMath/writ/CompensationAtUNCA/data/compression_ratios.csv'))

## Read the raw IPEDS export

In [2]:
raw = pd.read_csv(input_path, dtype=str)

# IPEDS exports sometimes include a trailing blank column.
raw = raw.loc[:, raw.columns.notna()]
raw = raw.loc[:, raw.columns.str.strip() != ""]

raw.head()

,UnitID,Institution Name,Sector of institution (HD2024),State abbreviation (HD2024),Carnegie Classification 2021: Basic (HD2024),Average salary for instructional staff equated to a 9-month contract-total (SAL2024_IS Professor),Average salary for instructional staff equated to a 9-month contract-total (SAL2024_IS Assistant professor),Total enrollment (DRVEF2024),Unnamed: 8
0,177834,A T Still University of Health Sciences,2,MO,25,122621,92847,3466,NaN
1,180203,Aaniiih Nakoda College,1,MT,33,NaN,NaN,120,NaN
2,222178,Abilene Christian University,2,TX,17,103820,75882,5219,NaN
3,497037,Abilene Christian University-Undergraduate Online,2,TX,-2,NaN,NaN,1224,NaN
4,138558,Abraham Baldwin Agricultural College,1,GA,23,76050,65819,3825,NaN


## Reshape salary columns into tidy rows

In [3]:
unit_col = "UnitID"
school_col = "Institution Name"
sector_col = "Sector of institution (HD2024)"
state_col = "State abbreviation (HD2024)"
classification_col = "Carnegie Classification 2021: Basic (HD2024)"
enrollment_col = "Total  enrollment (DRVEF2024)"

salary_cols = {
    "Average salary for instructional staff equated to a 9-month contract-total (SAL2024_IS  Professor)": "Full Professor",
    "Average salary for instructional staff equated to a 9-month contract-total (SAL2024_IS  Assistant professor)": "Assistant Professor",
}

required_cols = [
    unit_col,
    school_col,
    sector_col,
    state_col,
    classification_col,
    enrollment_col,
    *salary_cols.keys(),
]
missing_cols = [col for col in required_cols if col not in raw.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

tidy = raw[required_cols].melt(
    id_vars=[unit_col, school_col, sector_col, classification_col, enrollment_col, state_col],
    value_vars=list(salary_cols.keys()),
    var_name="SalaryMeasure",
    value_name="AvgSalary",
)

tidy["Rank"] = tidy["SalaryMeasure"].map(salary_cols)
tidy = tidy.drop(columns="SalaryMeasure")
tidy = tidy.rename(
    columns={
        school_col: "School",
        sector_col: "Sector",
        classification_col: "Classification",
        enrollment_col: "Enrollment",
        state_col: "State",
    }
)

tidy["AvgSalary"] = pd.to_numeric(tidy["AvgSalary"], errors="coerce")
tidy["Enrollment"] = pd.to_numeric(tidy["Enrollment"], errors="coerce").astype("Int64")
tidy = tidy.dropna(subset=["AvgSalary"])
tidy["AvgSalary"] = tidy["AvgSalary"].astype("Int64")
tidy["Rank"] = pd.Categorical(
    tidy["Rank"],
    categories=["Full Professor", "Assistant Professor"],
    ordered=True,
)

tidy = tidy[["UnitID", "School", "Rank", "AvgSalary", "Classification", "Enrollment", "State", "Sector"]]
tidy.head()

,UnitID,School,Rank,AvgSalary,Classification,Enrollment,State,Sector
0,177834,A T Still University of Health Sciences,Full Professor,122621,25,3466,MO,2
2,222178,Abilene Christian University,Full Professor,103820,17,5219,TX,2
4,138558,Abraham Baldwin Agricultural College,Full Professor,76050,23,3825,GA,1
10,384306,Acupuncture and Integrative Medicine College-B...,Full Professor,52500,26,115,CA,2
11,126182,Adams State University,Full Professor,89245,18,2901,CO,1


## Add readable Carnegie classification labels

The IPEDS data file stores the Carnegie classification as a code. When the accompanying value-labels file is present, this replaces the code with the human-readable classification label.

In [4]:
labels = pd.read_csv(labels_path, dtype=str)
classification_labels = labels[
    labels["VariableName"].eq(classification_col)
].set_index("Value")["ValueLabel"]
sector_labels = labels[
    labels["VariableName"].eq(sector_col)
].set_index("Value")["ValueLabel"]

tidy["Classification"] = tidy["Classification"].map(classification_labels).fillna(tidy["Classification"])
tidy["Sector"] = tidy["Sector"].map(sector_labels).fillna(tidy["Sector"])

tidy.head()

,UnitID,School,Rank,AvgSalary,Classification,Enrollment,State,Sector
0,177834,A T Still University of Health Sciences,Full Professor,122621,Special Focus Four-Year: Medical Schools & Cen...,3466,MO,"Private not-for-profit, 4-year or above"
2,222178,Abilene Christian University,Full Professor,103820,Doctoral/Professional Universities\r\n,5219,TX,"Private not-for-profit, 4-year or above"
4,138558,Abraham Baldwin Agricultural College,Full Professor,76050,Baccalaureate/Associate's Colleges: Mixed Bacc...,3825,GA,"Public, 4-year or above"
10,384306,Acupuncture and Integrative Medicine College-B...,Full Professor,52500,Special Focus Four-Year: Other Health Professi...,115,CA,"Private not-for-profit, 4-year or above"
11,126182,Adams State University,Full Professor,89245,Master's Colleges & Universities: Larger Programs,2901,CO,"Public, 4-year or above"


## Write the tidy CSV

In [5]:
tidy = tidy.sort_values(["School", "Rank"]).reset_index(drop=True)
tidy["Rank"] = tidy["Rank"].astype(str)
tidy.to_csv(output_path, index=False)

tidy.shape, output_path

((3867, 8),
 PosixPath('/Users/mcmcclur/GitHubDocuments/MarksMath/writ/CompensationAtUNCA/_process/ipeds_salary_compression_tidy.csv'))

## Write the compact compression-ratio CSV

This file has one row per institution with both salary values present. `CompressionRatio` is the Assistant Professor average salary divided by the Full Professor average salary, and `CarnegieClassification` preserves the numeric IPEDS code.

In [6]:
full_professor_col = "Average salary for instructional staff equated to a 9-month contract-total (SAL2024_IS  Professor)"
assistant_professor_col = "Average salary for instructional staff equated to a 9-month contract-total (SAL2024_IS  Assistant professor)"

compression = raw[
    [unit_col, school_col, sector_col, state_col, enrollment_col, classification_col, full_professor_col, assistant_professor_col]
].copy()

compression[full_professor_col] = pd.to_numeric(compression[full_professor_col], errors="coerce")
compression[assistant_professor_col] = pd.to_numeric(compression[assistant_professor_col], errors="coerce")
compression = compression.dropna(subset=[full_professor_col, assistant_professor_col])

compression["CompressionRatio"] = compression[assistant_professor_col] / compression[full_professor_col]
compression = compression.rename(
    columns={
        school_col: "School",
        sector_col: "Sector",
        state_col: "State",
        enrollment_col: "Enrollment",
        classification_col: "CarnegieClassification",
    }
)

compression["Enrollment"] = pd.to_numeric(compression["Enrollment"], errors="coerce").astype("Int64")
compression["CarnegieClassification"] = pd.to_numeric(
    compression["CarnegieClassification"], errors="coerce"
).astype("Int64")
compression["Sector"] = compression["Sector"].map(sector_labels).fillna(compression["Sector"])

compression = compression[
    ["UnitID", "School", "State", "Sector", "Enrollment", "CarnegieClassification", "CompressionRatio"]
]
compression = compression[(compression.Sector == "Public, 4-year or above") & (14 < compression.CarnegieClassification) & (compression.CarnegieClassification < 24)]
compression = compression.sort_values("School").reset_index(drop=True)
compression.to_csv(compression_output_path, index=False)
compression.to_csv(compression_data_output_path, index=False)

compression.shape, compression_output_path, compression_data_output_path

((570, 7),
 PosixPath('/Users/mcmcclur/GitHubDocuments/MarksMath/writ/CompensationAtUNCA/_process/compression_ratios.csv'),
 PosixPath('/Users/mcmcclur/GitHubDocuments/MarksMath/writ/CompensationAtUNCA/data/compression_ratios.csv'))

## Summarize compression ratios

In [7]:
compression_ratio_summary = compression["CompressionRatio"].agg(["mean", "std"])
compression_ratio_summary

mean    0.712140
std     0.083474
Name: CompressionRatio, dtype: float64

In [8]:
(0.812445 - compression_ratio_summary.values[0])/compression_ratio_summary.values[1]

np.float64(1.2016316655457464)

In [13]:
NCCompression = compression[compression.State == 'NC']
NCCompression['abbr'] = ['ASU', 'ECU', 'ECSU', 'FSU', 'NC A&T', 'NCCU', 'NCSU', 'UNCA', 'UNCW', 'UNC', 'UNCC', 'UNCG', 'UNCP', 'WCU', 'WSSU']
NCCompression.to_csv('NCCompression.csv', index=False)

/var/folders/dw/q0zlmbyn6qd55csnrk1ml4dh0000gp/T/ipykernel_77453/4153750562.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  NCCompression['abbr'] = ['ASU', 'ECU', 'ECSU', 'FSU', 'NC A&T', 'NCCU', 'NCSU', 'UNCA', 'UNCW', 'UNC', 'UNCC', 'UNCG', 'UNCP', 'WCU', 'WSSU']
